In [2]:
%reset -f

In [3]:
#Import necessary libraries
import base64
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [4]:
#Load the data
df = pd.read_excel('/Users/Vishnu/Desktop/Python/AI/FPL/fpl_history.xlsx')
print("File loaded successfully!")
backup_df = df.copy()


File loaded successfully!


In [5]:
#Run to Save
backup_df=df.copy()

In [6]:
#Run to reset
df=backup_df.copy()

In [7]:
#Cleaning the data
desired_columns = [
    'season_x', 'GW', 'name', 'position', 'team', 'opp_team_name',
    'minutes', 'assists', 'bonus', 'bps', 'goals_conceded', 'goals_scored',
    'total_points', 'saves', 'clean_sheets', 'own_goals', 'penalties_missed',
    'penalties_saved', 'yellow_cards', 'red_cards', 'creativity', 'threat',
    'influence', 'ict_index', 'team_a_score', 'team_h_score',
    'transfers_balance', 'selected', 'value', 'was_home', 'fixture']

df = df[df['season_x'] >= '2018-19']

df.drop(columns=['element', 'kickoff_time', 'opponent_team', 'round', 'transfers_in', 'fixture','transfers_out'], errors='ignore', inplace=True)


df = df.astype({'was_home': 'bool'})
df['position'] = df['position'].replace('GKP', 'GK')

df = df[[col for col in desired_columns if col in df.columns]]

print(df.head())


  season_x  GW                name position           team opp_team_name  \
0  2020-21   1      Aaron Connolly      FWD       Brighton       Chelsea   
1  2020-21   1     Aaron Cresswell      DEF       West Ham     Newcastle   
2  2020-21   1          Aaron Mooy      MID       Brighton       Chelsea   
3  2020-21   1      Aaron Ramsdale       GK  Sheffield Utd        Wolves   
4  2020-21   1  Abdoulaye Doucouré      MID        Everton         Spurs   

   minutes  assists  bonus  bps  ...  creativity  threat  influence  \
0       45        0      0   -3  ...         0.3      32        1.2   
1       90        0      0   11  ...        11.2       0       10.4   
2        0        0      0    0  ...         0.0       0        0.0   
3       90        0      0   12  ...         0.0       0       18.2   
4       90        0      0   20  ...        44.6       4       20.4   

   ict_index  team_a_score  team_h_score  transfers_balance  selected  value  \
0        3.4             3          

In [8]:
df['season_x'].unique()

array(['2020-21', '2021-22', '2022-23', '2023-24'], dtype=object)

In [9]:
#Team level aggregation
agg_dict = {
    'assists': 'sum',
    'bonus': 'sum',
    'bps': 'sum',
    'goals_conceded': 'max',
    'goals_scored': 'sum',
    'total_points': 'sum',
    'saves': 'sum',
    'clean_sheets': 'sum',
    'own_goals': 'sum',
    'penalties_missed': 'sum',
    'penalties_saved': 'sum',
    'yellow_cards': 'sum',
    'red_cards': 'sum',
    'creativity': 'sum',
    'threat': 'sum',
    'influence': 'sum',
    'ict_index': 'mean'  # average per player for interpretability
}

teamdf = (
    df.groupby(['season_x', 'GW', 'team'], as_index=False)
      .agg(agg_dict)
)

# Convert clean_sheets to boolean
teamdf['clean_sheets'] = teamdf['clean_sheets'] > 0
backup_teamdf = teamdf.copy()

In [10]:
teamdf = teamdf.sort_values(['season_x', 'team', 'GW']).reset_index(drop=True)

# 3- and 5-game rolling averages
teamdf['attack_form_3'] = teamdf.groupby('team')['goals_scored'].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
teamdf['attack_form_5'] = teamdf.groupby('team')['goals_scored'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
teamdf['defence_form_3'] = teamdf.groupby('team')['goals_conceded'].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
teamdf['defence_form_5'] = teamdf.groupby('team')['goals_conceded'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
teamdf['points_form_5'] = teamdf.groupby('team')['total_points'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))

In [11]:
# Attack/Defence/Team strength composites
teamdf['attack_score'] = teamdf['attack_form_5']
teamdf['defence_score'] = 1 / (1 + teamdf['defence_form_5'])
teamdf['team_strength'] = (teamdf['points_form_5'] + teamdf['attack_score']) / (1 + teamdf['defence_form_5'])

scaler = MinMaxScaler(feature_range=(0, 10))
cols_to_scale = ['attack_form_5', 'defence_form_5', 'points_form_5', 'attack_score', 'defence_score', 'team_strength']
teamdf[cols_to_scale] = scaler.fit_transform(teamdf[cols_to_scale])


In [12]:
teamdf[['attack_form_5', 'defence_form_5', 'points_form_5']].describe()

,attack_form_5,defence_form_5,points_form_5
count,2844.000000,2844.000000,2844.000000
mean,3.079876,3.719937,3.390547
std,1.560804,1.675846,1.486649
min,0.000000,0.000000,0.000000
25%,2.083333,2.500000,2.335329
50%,2.916667,3.500000,3.221557
75%,4.166667,4.500000,4.275449
max,10.000000,10.000000,10.000000


In [13]:
df = df.merge(
    teamdf[['season_x','GW','team','attack_score','defence_score','team_strength']],
    on=['season_x','GW','team'],
    how='left'
)

In [14]:
df.columns

Index(['season_x', 'GW', 'name', 'position', 'team', 'opp_team_name',
       'minutes', 'assists', 'bonus', 'bps', 'goals_conceded', 'goals_scored',
       'total_points', 'saves', 'clean_sheets', 'own_goals',
       'penalties_missed', 'penalties_saved', 'yellow_cards', 'red_cards',
       'creativity', 'threat', 'influence', 'ict_index', 'team_a_score',
       'team_h_score', 'transfers_balance', 'selected', 'value', 'was_home',
       'attack_score', 'defence_score', 'team_strength'],
      dtype='object')

In [15]:
df.to_csv('fpl_cleaned.csv', index=False)

In [16]:
def add_opponent_metrics(df, teamdf):
    """
    Adds opponent strength metrics to the dataframe.    """
    # Drop existing opponent columns if they exist
    opp_cols = ['opp_attack_score', 'opp_defence_score', 'opp_team_strength']
    df = df.drop(columns=[col for col in opp_cols if col in df.columns])
    
    # Create opponent metrics
    opp_metrics = teamdf[['season_x', 'GW', 'team', 'attack_score', 'defence_score', 'team_strength']].copy()
    opp_metrics = opp_metrics.rename(columns={
        'team': 'opp_team_name',
        'attack_score': 'opp_attack_score',
        'defence_score': 'opp_defence_score',
        'team_strength': 'opp_team_strength'
    })
    
    # Merge
    df = df.merge(opp_metrics, on=['season_x', 'GW', 'opp_team_name'], how='left')
    
    return df

# Use it like this:
df = add_opponent_metrics(df, teamdf).round(1)

In [17]:
# Step 2: Create fixture difficulty metrics
# For attackers: easier fixture = weaker opponent defense
df['fixture_difficulty_attack'] = df['attack_score'] - df['opp_defence_score']

# For defenders: easier fixture = weaker opponent attack  
df['fixture_difficulty_defence'] = df['defence_score'] - df['opp_attack_score']


# Overall fixture difficulty
df['fixture_difficulty_overall'] = df['team_strength'] - df['opp_team_strength']

In [18]:
scaler2 = MinMaxScaler(feature_range=(1.0, 5.0))

# List of fixture columns
fixture_cols = [
    'fixture_difficulty_attack',
    'fixture_difficulty_defence',
    'fixture_difficulty_overall'
]

# Scale and then invert (so 5 = hardest, 1 = easiest)
for col in fixture_cols:
    scaled = scaler2.fit_transform(df[[col]])
    df[col] = 6 - scaled.flatten().round(1)  # flatten to 1D and flip 1↔5 mapping


In [19]:
#Point form over 3 and 5 games
df['rolling_avg_3'] = df.groupby(['season_x', 'name'])['total_points'].transform(lambda x: x.rolling(window=3,min_periods=1).mean().shift(1))
df['rolling_avg_5'] = df.groupby(['season_x', 'name'])['total_points'].transform(lambda x: x.rolling(window=5,min_periods=1).mean().shift(1))

In [20]:
metric_groups = {
    'team': ['attack_score', 'defence_score', 'team_strength'],
    'opp_team_name': ['opp_attack_score', 'opp_defence_score', 'opp_team_strength'],
    'name': ['rolling_avg_3', 'rolling_avg_5'],
    'global': ['fixture_difficulty_attack', 'fixture_difficulty_defence', 'fixture_difficulty_overall']
}
for group_col, cols in metric_groups.items():
    for col in cols:
        if col in df.columns:
            if group_col == 'global':
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df.groupby([group_col, 'season_x'])[col].transform(lambda x: x.fillna(x.median()))
                df[col] = df[col].fillna(df[col].median())

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  